# 02 - Assessing & Cleaning

Notebook ini mendokumentasikan kualitas data, masalah yang ditemukan, dan keputusan cleaning yang diterapkan dalam pipeline `src_prepare_dataset.py`.

In [ ]:
import pandas as pd
from pathlib import Path

raw = pd.read_csv('../../KaggleV2-May-2016.csv', dtype={'PatientId': str})
raw.head()

## Assessing Data

Pemeriksaan awal meliputi missing value, duplikasi, tipe data, outlier usia, dan appointment dengan `waiting_days` negatif.

In [ ]:
raw.isna().sum()

In [ ]:
raw.duplicated().sum()

In [ ]:
raw.dtypes

In [ ]:
scheduled = pd.to_datetime(raw['ScheduledDay'], utc=True)
appointment = pd.to_datetime(raw['AppointmentDay'], utc=True)
waiting_days = (appointment.dt.normalize() - scheduled.dt.normalize()).dt.days
{
    'negative_age_rows': int((raw['Age'] < 0).sum()),
    'age_above_115_rows': int((raw['Age'] > 115).sum()),
    'negative_waiting_days_rows': int((waiting_days < 0).sum()),
}

## Temuan Kualitas Data

- Missing values: **0** nilai kosong.
- Duplicate rows: **0** baris duplikat.
- Age negatif: **1** baris.
- Waiting days negatif: **5** baris.
- Kolom `Hipertension` dan `Handcap` memiliki typo sehingga dinormalisasi menjadi `hypertension` dan `handicap`.

## Keputusan Cleaning

1. Nama kolom dinormalisasi ke snake_case agar konsisten.
2. `PatientId` disimpan sebagai string agar tidak kehilangan presisi.
3. `ScheduledDay` dan `AppointmentDay` dikonversi ke datetime UTC.
4. `No-show` diubah menjadi target binary `is_no_show`.
5. Baris dengan usia di luar rentang 0-115 dihapus.
6. Baris dengan `waiting_days` negatif dihapus karena appointment tidak boleh terjadi sebelum jadwal dibuat.
7. Fitur turunan dibuat: `waiting_days`, `scheduled_hour`, `appointment_weekday`, `age_group`, dan `has_chronic_condition`.

In [ ]:
%run ../src_prepare_dataset.py
clean = pd.read_csv('../data/processed/appointments_clean.csv')
clean.head()

In [ ]:
{'before': raw.shape, 'after': clean.shape, 'removed_rows': raw.shape[0] - clean.shape[0]}

In [ ]:
clean.isna().sum()

## Before/After Cleaning

Dataset awal memiliki **110,527** baris. Setelah cleaning, dataset final memiliki **110,521** baris. Total baris yang dibuang adalah **6**, terutama karena nilai usia atau `waiting_days` tidak valid.